<a href="https://colab.research.google.com/github/zelal-Eizaldeen/deeplearning_course/blob/main/Run_LLM_With_vLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run the LLM with a serving framework


Model serving frameworks—such as vLLM and SGLang—are purpose-built systems designed to load pre-trained language models and expose them through APIs (typically REST or gRPC) for real-time or batch inference. While training frameworks like PyTorch and TensorFlow focus on model development and gradient-based learning, serving frameworks are optimized for efficient, scalable, and low-latency inference.

A well-designed serving framework (like vLLM) also continually integrates the latest research optimizations—such as paged attention and speculative decoding—while abstracting away low-level infrastructure concerns. This allows developers to focus on building applications rather than re-implementing inference logic for each model architecture or chasing the latest academic papers about improvements.

Serving an LLMwith vLLM is quite simple and efficient. With just a few lines of code, you can load a model such as Qwen2.5 and run inference. Here’s a basic example:

In [1]:
import torch
import gc
import time

# Unload models and clean up gpu memory cache
def free_gpu(model):
  if model:
    # Removes the reference to the model's memory,
    # making it eligible for garbage collection.
    del model

  # Release any cached GPU memory that's no longer needed.
  if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

  # Trigger garbage collection to ensure memory is fully released.
  gc.collect()

free_gpu(None)



In [ ]:
!pip install vllm
!pip install transformers

Run Qwen model with vLLM and track the inference time.



In [ ]:
import time
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen2.5-0.5B"

# Load model with vLLM.
llm = LLM(model=model_name, dtype="float16")

# Define the prompt.
prompt = """You are an expert AI historian writing a detailed chapter for a book titled "The Evolution of Human-AI Collaboration."

Begin by summarizing the early stages of artificial intelligence in the 1950s, touching on symbolic logic and rule-based systems. Then transition into the rise of machine learning, particularly deep learning in the 2010s.

Afterward, describe how large language models like GPT transformed human-computer interaction, enabling applications in education, creative writing, customer support, and software development.

Finally, reflect on the societal and ethical implications of AI, such as misinformation, bias, and the alignment problem.

Write in a formal tone, with rich detail and examples in each era."""

# Create sampling parameters.
sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=128)

# Time the model generation.
start_time = time.time()
outputs = llm.generate([prompt], sampling_params)
end_time = time.time()

# Print the results.
for output in outputs:
  print(f"Generated text: {output}")
  print(f"Time taken: {end_time - start_time:.2f} seconds")

free_gpu(llm)

As you can see, the LLM() and generate() functions abstract away much of the complexity, enabling quick experimentation with LLMs.

In [8]:
free_gpu(llm)

# Performance Comparison: vLLM vs. Hugging Face Transformers

In addition to convenience, vLLM delivers substantial performance improvements. In many benchmarks, it achieves 10 to 20 times higher throughput than the Hugging Face .generate() API (transformer library).

Run Qwen model with standard (non-optimial) HuggingFace library and track the inference time.


In [ ]:
# --- Basic Model Serving (transformers) ---
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

start_time_basic = time.time()

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map="auto", trust_remote_code=True)

# Create the pipeline.
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

outputs_basic = generator(prompt, max_length=128, temperature=0.8, top_p=0.95)
end_time_basic = time.time()

print("\n---- Basic Model Serving Results ----")
for output in outputs_basic:
    print(f"Generated text: {output['generated_text']}")
    print(f"Time taken: {end_time_basic - start_time_basic:.2f} seconds")


print(f"\nLatency difference: {(end_time_basic - start_time_basic) - (end_time - start_time):.2f} seconds")

free_gpu(generator)


In [7]:
print(f"\nLatency difference: {(end_time_basic - start_time_basic) - (end_time - start_time):.2f} seconds")



Latency difference: 57.61 seconds


vLLM takes 1.04 seconds, while the Hugging Face Library takes 58.64 seconds. That’s a significant speedup on a single prompt.

Despite its simplicity, vLLM provides extensive configuration options to fine-tune performance and behavior to meet your specific application needs.